# 위대한 개츠비 책 전체 번역 — Gemma 4 31B (학습 없음)

설정은 11/17 `ctx5` 실험과 같음: 앞 문단 5개 문맥 · 생각 모드 끔 · greedy · 같은 지침.

**GPU에 따라 자동으로 갈림**
- 80GB급(A100 80GB, H100): 원본(bf16)으로 불러옴. 11/17 1·3장 결과를 이어 쓸 수 있음
- 40GB급(A100 40GB)·24GB급(L4): **4비트로 압축**해 불러옴. 원본과 조건이 달라 1·3장도 다시 번역함(올린 1·3장 파일은 쓰지 않음). 결과 파일 이름은 `gemma_full_4bit_ctx5.jsonl`. L4는 한 문단씩 처리해서 많이 느림
- T4(16GB)는 안 됨

책 전체 2,954문장(끝의 표제·목차 중복 5줄 제외). 원본(bf16)은 예상 2.5~3시간(1·3장 638문장에 34.6분 걸렸음. 1·3장 결과를 올리면 그만큼 줄어듦). 4비트는 이 책으로 돌려 본 적이 없어 시간·품질 모두 미확인 — 원본보다 느림.

**올릴 파일** (1번 셀 실행 시 파일 선택 창)
- 필수: `gemma_translate.py`, `sentences.jsonl`
- 80GB급일 때 권장: `gemma_ch1_3_bf16_ctx5.jsonl` (11/17 1·3장 결과) → 올리면 1·3장은 건너뜀. 이미 검증한 번역과 같은 문장을 유지하려면 꼭 올릴 것. 4비트 실행에서는 올려도 쓰지 않음

**실행**: ① 설치 셀 → '런타임 → 세션 다시 시작' → 1번 셀부터 끝까지.
결과는 구글 드라이브 `MyDrive/bridgecore_gatsby/`에 계속 저장됨. 세션이 끊기면 1번부터 다시 실행(파일은 다시 안 올려도 됨) → 남은 문단만 이어서 번역.

드라이브를 못 쓰는 계정이면 자동으로 세션 안(`/content/bridgecore_gatsby/`)에 저장함. 이 경우 세션이 끊기면 결과가 사라지므로, 왼쪽 파일 탭에서 `out/gemma_full_bf16_ctx5.jsonl`(4비트면 `gemma_full_4bit_ctx5.jsonl`)을 중간에 한두 번 내려받아 둘 것. 끊긴 뒤에는 그 파일을 1번 셀 파일 선택 창에서 같이 올리면 이어서 번역함.

**결과 파일**: 원본이면 `gemma_full_bf16_ctx5.jsonl`·`run_info_full_ctx5.json`, 4비트면 `gemma_full_4bit_ctx5.jsonl`·`run_info_full_ctx5_4bit.json` (끝나면 자동 다운로드)


In [ ]:
# ① 맨 먼저 이 셀만 실행 → 끝나면 '런타임 → 세션 다시 시작' → 다음 셀부터
!pip list --format=freeze | grep -E '^(torch|torchvision|torchaudio)==' > /content/torch_pin.txt
!pip -q install -c /content/torch_pin.txt -U "transformers>=5.10.1" accelerate
!pip -q install -U "bitsandbytes>=0.46.1"
!pip -q uninstall -y torchao
!pip list --format=freeze | grep -E '^(transformers|torch|accelerate)=='


## 1. 설정·파일 올리기

In [ ]:
import json, sys, time, platform, inspect, re, shutil
from pathlib import Path
import torch
from google.colab import files, drive

GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f'{GPU_GB:.0f}GB')
assert GPU_GB > 20, 'GPU 메모리가 20GB보다 작습니다(T4 등) — 4비트로도 31B 모델이 올라가지 않습니다. L4·A100·H100으로 바꾸세요'
LOAD_4BIT = GPU_GB < 70          # 80GB급이면 원본(bf16), 그보다 작으면 4비트 압축
NAME = 'full_ctx5_4bit' if LOAD_4BIT else 'full_ctx5'
RUNTIME = 'colab_transformers_4bit' if LOAD_4BIT else 'colab_transformers_bf16'
CFG = dict(n_ctx=5, thinking=False)
CONFIGS = {NAME: CFG}
MODEL_ID = 'google/gemma-4-31B-it'
BS = 4 if GPU_GB > 35 else 1     # 24GB급은 한 문단씩
THINK_BUDGET = 4096
print('불러오는 방식:', '4비트 압축 — 1·3장도 다시 번역합니다(11/17 원본 결과와 조건이 달라서)' if LOAD_4BIT else '원본(bf16)')

try:
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/bridgecore_gatsby')
except Exception as e:      # 드라이브를 못 쓰는 계정(공용 계정 등) → 세션 안에만 저장
    BASE = Path('/content/bridgecore_gatsby')
    print('드라이브 연결 안 됨 → 세션 안에 저장합니다. 세션이 끊기면 결과가 사라지니 4번 셀의 중간 저장 파일을 가끔 내려받으세요:', e)
OUT_DIR = BASE / 'out'; OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / ('gemma_full_4bit_ctx5.jsonl' if LOAD_4BIT else 'gemma_full_bf16_ctx5.jsonl')
need = [n for n in ('gemma_translate.py', 'sentences.jsonl') if not (BASE / n).exists()]
if need or not OUT.exists():
    print('파일 선택 창에서 고르기:', need or '(필수 파일은 이미 있음)', '+ 권장 gemma_ch1_3_bf16_ctx5.jsonl — 없으면 취소')
    for name, data in files.upload().items():
        clean = re.sub(r' \(\d+\)(?=\.[^.]+$)', '', Path(name).name)
        if clean.endswith('.jsonl') and clean != 'sentences.jsonl':
            (OUT_DIR / 'prev_ch1_3.jsonl').write_bytes(data)
        else:
            (BASE / clean).write_bytes(data)

assert (BASE / 'sentences.jsonl').exists() and (BASE / 'gemma_translate.py').exists(), '필수 파일 두 개를 올리세요'
sys.path.insert(0, str(BASE))
import gemma_translate as gt
assert 'n_ctx' in inspect.signature(gt.user_text).parameters, 'gemma_translate.py가 옛 버전입니다'
assert gt.PROMPT_HASH == '4f5cbcadc8d7', f'지침이 11/17과 다릅니다({gt.PROMPT_HASH}) — 같은 gemma_translate.py를 올리세요'

PREV = OUT_DIR / 'prev_ch1_3.jsonl'
if PREV.exists() and not OUT.exists():
    rows_prev = [json.loads(l) for l in open(PREV, encoding='utf-8') if l.strip()]
    ok_cfg = (NAME,) if LOAD_4BIT else ('ctx5', NAME)    # 4비트 실행은 4비트 중간 저장본만 이어받음
    if all(r['mt_meta'].get('config') in ok_cfg and r['mt_meta'].get('prompt_hash') == gt.PROMPT_HASH for r in rows_prev):
        shutil.copy(PREV, OUT); print('기존 결과', len(rows_prev), '문장을 이어 씀')
    else:
        print('올린 jsonl은 이번 실행 조건과 달라 쓰지 않습니다(4비트 실행에 원본 1·3장 결과를 올린 경우 등) → 전부 새로 번역')
print('프롬프트', gt.PROMPT_HASH, '· 이미 번역된 문장', sum(1 for _ in open(OUT, encoding='utf-8')) if OUT.exists() else 0)


## 2. 모델 불러오기 (약 5분)

In [ ]:
from google.colab import userdata
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
import transformers
from getpass import getpass

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None   # 공용 계정이면 열쇠 아이콘에 저장하지 말고, 필요할 때만 아래 줄 주석을 풀어 입력
# HF_TOKEN = getpass('HF 토큰: ')

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
processor.tokenizer.padding_side = 'left'
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16,
                           bnb_4bit_use_double_quant=True) if LOAD_4BIT else None
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map='cuda', token=HF_TOKEN,
                                             quantization_config=quant).eval()
LOAD_SEC = time.time() - t0
print(f'불러오기 {LOAD_SEC/60:.1f}분 · GPU {torch.cuda.memory_allocated()/1e9:.1f}GB')

rows = [json.loads(l) for l in open(BASE / 'sentences.jsonl', encoding='utf-8') if l.strip()]
allb, idx = gt.blocks(rows, None)   # 책 전체
WANT = {s['id'] for k in idx for s in allb[k]['sents'] if s['translate']}
print('문단', len(idx), '· 문장', len(WANT))


## 3. 번역 함수 (06 노트북과 동일)

In [ ]:
SAMPLE = dict(do_sample=True, temperature=1.0, top_p=0.95, top_k=64)   # 공식 권장값 — greedy가 실패한 문단 재시도에만 씀

def prompt_text(k, cfg):
    sents, user = gt.user_text(allb, k, n_ctx=cfg['n_ctx'])
    msgs = [{'role': 'system', 'content': gt.SYSTEM + gt.FORMAT}, {'role': 'user', 'content': user}]
    return sents, processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                                enable_thinking=cfg['thinking'])

def split_answer(ids):
    raw = processor.decode(ids, skip_special_tokens=False)
    try:
        r = processor.parse_response(raw)
        return raw, r.get('content') or '', r.get('thinking') or ''
    except Exception:
        return raw, processor.decode(ids, skip_special_tokens=True), ''

@torch.inference_mode()
def generate(ks, cfg, sample=False):
    packs = [prompt_text(k, cfg) for k in ks]
    enc = processor(text=[p[1] for p in packs], return_tensors='pt', padding=True).to(model.device)
    n_in = enc['input_ids'].shape[1]
    need = max(len(p[0]) for p in packs) * 200 + 300 + (THINK_BUDGET if cfg['thinking'] else 0)
    out = model.generate(**enc, max_new_tokens=need, **(SAMPLE if sample else dict(do_sample=False)))
    res = []
    for (sents, _), ids in zip(packs, out[:, n_in:]):
        raw, content, think = split_answer(ids)
        n_tok = int((ids != processor.tokenizer.pad_token_id).sum())
        try:
            got = gt.parse(content)
            miss = [s['id'] for s in sents if not got.get(s['id'])]
            err = f'id 누락 {miss}' if miss else None
        except Exception as e:
            got, err = {}, f'파싱 실패 {e.__class__.__name__}'
        res.append(dict(sents=sents, got=got, err=err, raw=raw, think_chars=len(think), out_tokens=n_tok))
    return res

def run_config(name, cfg, out_path, only=None):
    meta = {'model': MODEL_ID, 'runtime': RUNTIME, 'prompt_hash': gt.PROMPT_HASH, 'config': name,
            'n_ctx': cfg['n_ctx'], 'thinking': cfg['thinking'], 'decoding': 'greedy',
            'transformers': transformers.__version__, 'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0)}
    done = {json.loads(l)['id'] for l in open(out_path, encoding='utf-8')} if out_path.exists() else set()
    todo = [k for k in (only if only is not None else idx)
            if any(s['translate'] and s['id'] not in done for s in allb[k]['sents'])]
    order = sorted(todo, key=lambda k: len(gt.user_text(allb, k, n_ctx=cfg['n_ctx'])[1]))
    stats = {'think_chars': [], 'out_tokens': [], 'retried_with_sampling': [], 'fails': {}}
    raw_log = open(OUT_DIR / f'raw_fail_{name}.jsonl', 'a', encoding='utf-8')

    def save(r, sampled=False):
        with open(out_path, 'a', encoding='utf-8') as f:
            for s in r['sents']:
                if s['id'] in done:      # 이미 있는 문장은 다시 쓰지 않음(중복 방지)
                    continue
                f.write(json.dumps({'id': s['id'], 'en': s['en'], 'mt': r['got'][s['id']],
                                    'mt_system': f'gemma-4-31b-it-bf16-{name}',
                                    'mt_meta': {**meta, 'decoding': 'sampling_retry' if sampled else 'greedy'}},
                                   ensure_ascii=False) + '\n')

    t0, bs, i, fails, nb = time.time(), BS, 0, [], 0
    print(f'[{name}] 문단 {len(order)}개 시작')
    while i < len(order):
        ks = order[i:i + bs]
        try:
            res = generate(ks, cfg)
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs == 1:
                raise
            bs = max(1, bs // 2); print('  메모리 부족 → 묶음', bs); continue
        for k, r in zip(ks, res):
            stats['think_chars'].append(r['think_chars']); stats['out_tokens'].append(r['out_tokens'])
            if r['err']:
                fails.append(k); raw_log.write(json.dumps({'k': k, 'err': r['err'], 'raw': r['raw']}, ensure_ascii=False) + '\n')
            else:
                save(r)
        i += len(ks); nb += 1
        if nb % 5 == 0 or i >= len(order):
            el = time.time() - t0
            print(f'  {i}/{len(order)} 문단 · {el/60:.1f}분 · 남은 예상 {el/i*(len(order)-i)/60:.0f}분 · 실패 {len(fails)}')
    for k in fails:                      # greedy로 실패한 문단: 하나씩, 공식 권장 샘플링으로 한 번 더
        r = generate([k], cfg, sample=True)[0]
        if r['err']:
            stats['fails'][str(k)] = r['err']
            raw_log.write(json.dumps({'k': k, 'err': r['err'], 'raw': r['raw'], 'sampling': True}, ensure_ascii=False) + '\n')
        else:
            save(r, sampled=True); stats['retried_with_sampling'].append(k)
    raw_log.close()
    got_ids = {json.loads(l)['id'] for l in open(out_path, encoding='utf-8')} if out_path.exists() else set()
    tc = [x for x in stats['think_chars'] if x]
    info = {**meta, 'gen_sec': round(time.time() - t0), 'n_sentences': len(got_ids & WANT), 'missing': sorted(WANT - got_ids),
            'retried_with_sampling': stats['retried_with_sampling'], 'fails': stats['fails'], 'batch_final': bs,
            'mean_out_tokens': round(sum(stats['out_tokens']) / max(1, len(stats['out_tokens']))),
            'mean_think_chars': round(sum(tc) / len(tc)) if tc else 0, 'load_sec': round(LOAD_SEC),
            'python': platform.python_version()}
    (OUT_DIR / f'run_info_{name}.json').write_text(json.dumps(info, ensure_ascii=False, indent=1))
    print(f'[{name}] 완료 {info["gen_sec"]/60:.1f}분 · {info["n_sentences"]}/{len(WANT)}문장 · 빠짐 {len(info["missing"])}'
          f' · 샘플링 재시도 성공 {len(stats["retried_with_sampling"])} · 평균 생각 {info["mean_think_chars"]}자')
    return info


# 시험 1문단 — 형식 확인 (저장하지 않음)
r = generate(idx[:1], CFG)[0]
print('오류', r['err'], '· 출력 토큰', r['out_tokens'])
for s in r['sents']:
    print('EN', s['en']); print('KO', r['got'].get(s['id']))


## 4. 책 전체 번역 (드라이브에 계속 저장, 끊기면 다시 실행해 이어서)

In [ ]:
info = run_config(NAME, CFG, OUT)
rows_out = [json.loads(l) for l in open(OUT, encoding='utf-8')]
ids = [r['id'] for r in rows_out]
assert len(ids) == len(set(ids)), '중복 문장이 있습니다'
print('번역', len(ids), '/', len(WANT), '· 빠짐', len(WANT - set(ids)), '· 빈 번역', sum(1 for r in rows_out if not (r['mt'] or '').strip()))
files.download(str(OUT)); files.download(str(OUT_DIR / f'run_info_{NAME}.json'))
fail = OUT_DIR / f'raw_fail_{NAME}.jsonl'
if fail.exists() and fail.stat().st_size:
    files.download(str(fail))
